# Workflow Orchestration

Workflow orchestration is the coordination of multiple data processing tasks in a specific order. This notebook explores workflow patterns and scheduling concepts.

## What is Workflow Orchestration?

Orchestration involves:
- **Task Dependencies**: Define which tasks depend on others
- **Scheduling**: Run workflows at specific times
- **Monitoring**: Track task execution and failures
- **Retry Logic**: Automatically retry failed tasks
- **Parallel Execution**: Run independent tasks concurrently

## Simple Task Dependencies

Let's create a simple workflow with dependencies:

In [ ]:
from datetime import datetime, timedelta
import time
import random

class Task:
    """Represents a single task in a workflow"""
    
    def __init__(self, name, function, dependencies=None):
        self.name = name
        self.function = function
        self.dependencies = dependencies or []
        self.status = 'pending'
        self.result = None
        self.start_time = None
        self.end_time = None
    
    def run(self):
        """Execute the task"""
        self.status = 'running'
        self.start_time = datetime.now()
        print(f"[{self.start_time.strftime('%H:%M:%S')}] Running task: {self.name}")
        
        try:
            self.result = self.function()
            self.status = 'success'
            self.end_time = datetime.now()
            duration = (self.end_time - self.start_time).total_seconds()
            print(f"[{self.end_time.strftime('%H:%M:%S')}] ✓ {self.name} completed in {duration:.2f}s")
        except Exception as e:
            self.status = 'failed'
            self.end_time = datetime.now()
            print(f"[{self.end_time.strftime('%H:%M:%S')}] ✗ {self.name} failed: {str(e)}")
            raise
    
    def can_run(self, completed_tasks):
        """Check if all dependencies are completed"""
        return all(dep in completed_tasks for dep in self.dependencies)

print("Task class defined")

## Workflow Manager

Create a workflow manager to orchestrate tasks:

In [ ]:
class WorkflowManager:
    """Manages workflow execution"""
    
    def __init__(self, name):
        self.name = name
        self.tasks = {}
        self.completed = set()
        self.failed = set()
    
    def add_task(self, task):
        """Add a task to the workflow"""
        self.tasks[task.name] = task
    
    def run(self):
        """Execute the workflow"""
        print(f"\n{'=' * 60}")
        print(f"Starting Workflow: {self.name}")
        print(f"{'=' * 60}\n")
        
        start_time = datetime.now()
        pending_tasks = set(self.tasks.keys())
        
        while pending_tasks:
            # Find tasks that can run
            runnable = [
                task_name for task_name in pending_tasks
                if self.tasks[task_name].can_run(self.completed)
            ]
            
            if not runnable:
                print("\n✗ No runnable tasks found. Workflow deadlocked.")
                break
            
            # Run all runnable tasks
            for task_name in runnable:
                task = self.tasks[task_name]
                try:
                    task.run()
                    self.completed.add(task_name)
                except Exception:
                    self.failed.add(task_name)
                
                pending_tasks.remove(task_name)
                print()  # blank line for readability
        
        end_time = datetime.now()
        total_duration = (end_time - start_time).total_seconds()
        
        # Print summary
        print(f"{'=' * 60}")
        print(f"Workflow Summary: {self.name}")
        print(f"{'=' * 60}")
        print(f"Total Duration: {total_duration:.2f}s")
        print(f"Completed Tasks: {len(self.completed)}")
        print(f"Failed Tasks: {len(self.failed)}")
        
        if self.failed:
            print(f"\n✗ Workflow completed with failures: {', '.join(self.failed)}")
        else:
            print(f"\n✓ Workflow completed successfully!")
        print(f"{'=' * 60}\n")

print("WorkflowManager class defined")

## Example Workflow

Create a data pipeline workflow with dependencies:

In [ ]:
# Define task functions
def extract_data():
    """Extract data from source"""
    time.sleep(1)  # Simulate work
    return {'records': 100}

def validate_data():
    """Validate extracted data"""
    time.sleep(0.5)  # Simulate work
    return {'valid': True}

def transform_data():
    """Transform data"""
    time.sleep(1.5)  # Simulate work
    return {'transformed': 100}

def generate_report():
    """Generate summary report"""
    time.sleep(0.5)  # Simulate work
    return {'report': 'summary.pdf'}

def load_to_warehouse():
    """Load data to warehouse"""
    time.sleep(1)  # Simulate work
    return {'loaded': True}

def send_notification():
    """Send completion notification"""
    time.sleep(0.3)  # Simulate work
    return {'sent': True}

# Create workflow
workflow = WorkflowManager("Data Pipeline")

# Add tasks with dependencies
workflow.add_task(Task("extract", extract_data))
workflow.add_task(Task("validate", validate_data, dependencies=["extract"]))
workflow.add_task(Task("transform", transform_data, dependencies=["validate"]))
workflow.add_task(Task("report", generate_report, dependencies=["transform"]))
workflow.add_task(Task("load", load_to_warehouse, dependencies=["transform"]))
workflow.add_task(Task("notify", send_notification, dependencies=["report", "load"]))

# Run the workflow
workflow.run()

## Parallel Execution

Tasks without dependencies can run in parallel:

In [ ]:
import concurrent.futures

class ParallelWorkflowManager:
    """Workflow manager with parallel execution"""
    
    def __init__(self, name, max_workers=4):
        self.name = name
        self.tasks = {}
        self.completed = set()
        self.failed = set()
        self.max_workers = max_workers
    
    def add_task(self, task):
        self.tasks[task.name] = task
    
    def run(self):
        """Execute workflow with parallel execution"""
        print(f"\n{'=' * 60}")
        print(f"Starting Parallel Workflow: {self.name}")
        print(f"{'=' * 60}\n")
        
        start_time = datetime.now()
        pending_tasks = set(self.tasks.keys())
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            while pending_tasks:
                # Find tasks that can run
                runnable = [
                    task_name for task_name in pending_tasks
                    if self.tasks[task_name].can_run(self.completed)
                ]
                
                if not runnable:
                    break
                
                # Submit tasks to executor
                future_to_task = {
                    executor.submit(self.tasks[name].run): name 
                    for name in runnable
                }
                
                # Wait for tasks to complete
                for future in concurrent.futures.as_completed(future_to_task):
                    task_name = future_to_task[future]
                    try:
                        future.result()
                        self.completed.add(task_name)
                    except Exception:
                        self.failed.add(task_name)
                    
                    pending_tasks.remove(task_name)
        
        end_time = datetime.now()
        total_duration = (end_time - start_time).total_seconds()
        
        print(f"\n{'=' * 60}")
        print(f"Parallel Workflow Summary")
        print(f"{'=' * 60}")
        print(f"Total Duration: {total_duration:.2f}s")
        print(f"Completed: {len(self.completed)}, Failed: {len(self.failed)}")
        print(f"{'=' * 60}\n")

# Create parallel workflow
parallel_workflow = ParallelWorkflowManager("Parallel Pipeline", max_workers=3)

# Add tasks
parallel_workflow.add_task(Task("extract", extract_data))
parallel_workflow.add_task(Task("validate", validate_data, dependencies=["extract"]))
parallel_workflow.add_task(Task("transform", transform_data, dependencies=["validate"]))
parallel_workflow.add_task(Task("report", generate_report, dependencies=["transform"]))
parallel_workflow.add_task(Task("load", load_to_warehouse, dependencies=["transform"]))

# Run parallel workflow
parallel_workflow.run()

## Retry Logic

Add retry logic for failed tasks:

In [ ]:
class RetryableTask(Task):
    """Task with retry capability"""
    
    def __init__(self, name, function, dependencies=None, max_retries=3):
        super().__init__(name, function, dependencies)
        self.max_retries = max_retries
        self.retry_count = 0
    
    def run(self):
        """Execute with retry logic"""
        while self.retry_count <= self.max_retries:
            try:
                self.status = 'running'
                self.start_time = datetime.now()
                
                if self.retry_count > 0:
                    print(f"[{self.start_time.strftime('%H:%M:%S')}] Retry {self.retry_count}/{self.max_retries}: {self.name}")
                else:
                    print(f"[{self.start_time.strftime('%H:%M:%S')}] Running task: {self.name}")
                
                self.result = self.function()
                self.status = 'success'
                self.end_time = datetime.now()
                duration = (self.end_time - self.start_time).total_seconds()
                print(f"[{self.end_time.strftime('%H:%M:%S')}] ✓ {self.name} completed in {duration:.2f}s")
                return
                
            except Exception as e:
                self.retry_count += 1
                if self.retry_count > self.max_retries:
                    self.status = 'failed'
                    self.end_time = datetime.now()
                    print(f"[{self.end_time.strftime('%H:%M:%S')}] ✗ {self.name} failed after {self.max_retries} retries")
                    raise
                else:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠ {self.name} failed, retrying...")
                    time.sleep(1)  # Wait before retry

# Function that may fail
def unreliable_task():
    """Task that randomly fails"""
    time.sleep(0.5)
    if random.random() < 0.7:  # 70% chance of failure
        raise Exception("Random failure")
    return {'success': True}

# Test retry logic
retry_workflow = WorkflowManager("Retry Test")
retry_workflow.add_task(RetryableTask("unreliable", unreliable_task, max_retries=3))
retry_workflow.run()

## Scheduling Patterns

Common scheduling patterns in data engineering:

In [ ]:
class SchedulePattern:
    """Define scheduling patterns"""
    
    @staticmethod
    def daily(hour=0, minute=0):
        """Schedule daily at specific time"""
        return f"Run daily at {hour:02d}:{minute:02d}"
    
    @staticmethod
    def hourly(minute=0):
        """Schedule every hour"""
        return f"Run hourly at minute {minute}"
    
    @staticmethod
    def cron(expression):
        """Schedule using cron expression"""
        return f"Cron: {expression}"
    
    @staticmethod
    def interval(minutes):
        """Schedule at fixed interval"""
        return f"Run every {minutes} minutes"

# Examples
print("Scheduling Examples:")
print("1.", SchedulePattern.daily(hour=2, minute=30))
print("2.", SchedulePattern.hourly(minute=15))
print("3.", SchedulePattern.cron("0 */6 * * *"))  # Every 6 hours
print("4.", SchedulePattern.interval(30))  # Every 30 minutes

## Workflow Monitoring

Track and monitor workflow execution:

In [ ]:
class WorkflowMonitor:
    """Monitor workflow execution"""
    
    def __init__(self):
        self.runs = []
    
    def record_run(self, workflow_name, status, duration, tasks_completed, tasks_failed):
        """Record workflow run"""
        run_info = {
            'timestamp': datetime.now(),
            'workflow': workflow_name,
            'status': status,
            'duration': duration,
            'tasks_completed': tasks_completed,
            'tasks_failed': tasks_failed
        }
        self.runs.append(run_info)
    
    def get_statistics(self):
        """Get workflow statistics"""
        if not self.runs:
            return "No runs recorded"
        
        total_runs = len(self.runs)
        successful = sum(1 for r in self.runs if r['status'] == 'success')
        failed = total_runs - successful
        avg_duration = sum(r['duration'] for r in self.runs) / total_runs
        
        return {
            'total_runs': total_runs,
            'successful': successful,
            'failed': failed,
            'success_rate': f"{(successful/total_runs)*100:.1f}%",
            'avg_duration': f"{avg_duration:.2f}s"
        }

# Example usage
monitor = WorkflowMonitor()
monitor.record_run("Data Pipeline", "success", 5.2, 6, 0)
monitor.record_run("Data Pipeline", "success", 5.5, 6, 0)
monitor.record_run("Data Pipeline", "failed", 3.1, 4, 2)

print("\nWorkflow Statistics:")
stats = monitor.get_statistics()
for key, value in stats.items():
    print(f"  {key}: {value}")

## Popular Orchestration Tools

### Apache Airflow
- Most popular orchestration platform
- DAG-based workflows
- Rich UI for monitoring
- Extensive integrations

### Prefect
- Modern alternative to Airflow
- Python-native workflows
- Better error handling
- Cloud-native

### Dagster
- Data-aware orchestration
- Type-checked pipelines
- Built-in testing
- Asset-based lineage

### Luigi
- Simpler than Airflow
- Good for batch processing
- Dependency resolution

## Exercises

1. Create a workflow with conditional branching
2. Implement a circuit breaker for failing tasks
3. Add metrics collection to workflow execution
4. Create a workflow that can pause and resume
5. Implement workflow versioning

## Best Practices

1. **Idempotency**: Tasks should be idempotent
2. **Atomicity**: Each task should be atomic
3. **Clear Dependencies**: Make dependencies explicit
4. **Error Handling**: Handle failures gracefully
5. **Monitoring**: Track all workflow metrics
6. **Documentation**: Document workflow logic
7. **Testing**: Test workflows thoroughly